# Analyzing and Visualizing Stock Put Option Data as a Scatter Plot with Plotly Express

### Python Libraries Used

In [13]:
from tkinter import Tk
from tkinter.filedialog import askopenfilename
import pandas as pd
import plotly.express as px

### Accessing the CSV file using Tkinter
- In the past, I have used the yfinance API to avoid downloading CSV files.  While researching how to access a CSV file through python, I stumbled on this super useful library!
I'm a little late to party ;)

In [14]:
# Creating Tkinter root window
Tk().withdraw()

# Open Finder to select one of the CSV files
file_path = askopenfilename(title="Select CSV file", filetypes=[("CSV Files", "*.csv")])

# Read the CSV file into a DataFrame
df = pd.read_csv(file_path)

2024-12-06 22:07:25.059 Python[24577:14946680] WARNING: Secure coding is automatically enabled for restorable state! However, not on all supported macOS versions of this application. Opt-in to secure coding explicitly by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState:.


### Cleaning and Filtering the Data Set
- While originally I had intended to leave the CSV file unfiltered, the option contracts with Implied Volatility values over 100% were skewing the color gradient to far.  I chose to filter out those contracts since they weren't critical to my analysis

In [15]:
# Removing commas and percent signs and converting integers to floats
df['OTM Prob'] = df['OTM Prob'].str.replace('%', '').astype(float)
df['Ann Rtn'] = df['Ann Rtn'].str.replace(',', '').replace({r'%': ''}, regex=True).astype(float)
df['IV'] = df['IV'].str.replace('%', '').astype(float)
df['Moneyness'] = df['Moneyness'].str.replace('%', '').str.replace('-', '').astype(float)

# Deleting unnecessary columns from the data frame
columns_to_delete = ['Time', 'Bid', 'BE (Bid)']
df.drop(columns=columns_to_delete, inplace=True, errors='ignore')
df = df.dropna(subset=['Moneyness'])

# Filtering out Options with an Implied Volotality over 100%
filtered_df = df[(df['IV'] < 100)]
df = filtered_df

### Plotting the Data
- I chose to use 'Out of the Money' Probability and Annualized Return as my X and Y axis to visualize the risk vs reward profile of the option contracts in the CSV file

- I used 'Moneyness' and Implied Volatility as the size and color of the individual data points on the plot for to compare contracts that are close to each other

    - A common saying in the investing community is that everything is priced in.  I knew the saying was correct but didn't truly understood the concept until I zoomed in on this scatter plot.

In [16]:
fig = px.scatter(
    df,
    x='OTM Prob',
    y='Ann Rtn',
    title='Option Contracts Scatter Plot',
    labels={
        'OTM Prob': 'Out of the Money Probability',
        'Ann Rtn': 'Annualized Return (%)',
    },
    hover_data=df.columns,
    # Adding a trendline using OLS regression
    trendline='ols',
    size='Moneyness',
    color='IV'
)

fig.update_layout(
    xaxis=dict(title='OTM Probability %'),
    yaxis=dict(title='Annual Return %'),
    template='plotly_dark'
)

fig.show()

### Interactive Scatter Plot